In [1]:
import pandas as pd

df = pd.read_csv('khk.noun.tsv', sep = '\t')
df

,prompt,answer
0,багтраагаа,багтраа <voc>
1,дэнсээс,дэнс <abl>
2,тархалтыг,тархалт <acc>
3,дуулийг,дууль <acc>
4,гуалингаар,гуалин <ins>
...,...,...
14391,үтрээний,үтрээ <gen>
14392,лавлагаанд,лавлагаа <dat>
14393,эзэмшлээ,эзэмшил <voc>
14394,улстай,улс <com>


In [3]:
import datasets
from datasets import Dataset, DatasetDict

df['prompt'] = '<s> bb: '+ df['prompt']
df['answer'] = df['answer']+'</s>'

infl_dataset = Dataset.from_pandas(df)
infl_dataset

Dataset({
    features: ['prompt', 'answer'],
    num_rows: 14396
})

In [5]:
ds_train_devtest = infl_dataset.train_test_split(test_size=0.025, seed = 42)
ds_devtest = ds_train_devtest['test'].train_test_split(test_size=0.5, seed = 42)

ds_splits = DatasetDict({
    'train': ds_train_devtest['train'],
    'valid': ds_devtest['train'],
    'test': ds_devtest['test']
})

ds_splits

DatasetDict({
    train: Dataset({
        features: ['prompt', 'answer'],
        num_rows: 14036
    })
    valid: Dataset({
        features: ['prompt', 'answer'],
        num_rows: 180
    })
    test: Dataset({
        features: ['prompt', 'answer'],
        num_rows: 180
    })
})

In [6]:
def concatenate_columns(example):
    example["prompt"] = example["prompt"] + " " + example["answer"]
    return example

ds_splits["train"] = ds_splits["train"].map(concatenate_columns)
ds_splits["valid"] = ds_splits["valid"].map(concatenate_columns)

Map:   0%|          | 0/14036 [00:00<?, ? examples/s]

Map:   0%|          | 0/180 [00:00<?, ? examples/s]

In [7]:
ds_splits["train"][0:10]

{'prompt': ['<s> bb: <s> bb: хавчаарт хавчаар <dat></s></s>',
  '<s> bb: <s> bb: санхүүд санхүү <dat></s></s>',
  '<s> bb: <s> bb: зохицлын зохицол <gen></s></s>',
  '<s> bb: <s> bb: сейфээ сейф <voc></s></s>',
  '<s> bb: <s> bb: уулаа уул <voc></s></s>',
  '<s> bb: <s> bb: эхнэртэй эхнэр <com></s></s>',
  '<s> bb: <s> bb: ордноос ордон <abl></s></s>',
  '<s> bb: <s> bb: оноонд оноо <dat></s></s>',
  '<s> bb: <s> bb: нөөцийг нөөц <acc></s></s>',
  '<s> bb: <s> bb: цогцсыг цогцос <acc></s></s>'],
 'answer': ['хавчаар <dat></s></s>',
  'санхүү <dat></s></s>',
  'зохицол <gen></s></s>',
  'сейф <voc></s></s>',
  'уул <voc></s></s>',
  'эхнэр <com></s></s>',
  'ордон <abl></s></s>',
  'оноо <dat></s></s>',
  'нөөц <acc></s></s>',
  'цогцос <acc></s></s>']}

In [8]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bayartsogt/mongolian-gpt2")

tokenizer_config.json:   0%|          | 0.00/207 [00:00<?, ?B/s]

d:\Anaconda3\envs\pt_env\Lib\site-packages\huggingface_hub\file_download.py:148: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\mnv\.cache\huggingface\hub\models--bayartsogt--mongolian-gpt2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to see activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


vocab.json:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.33M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/3.10M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/24.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/90.0 [00:00<?, ?B/s]

In [9]:
ds_splits = ds_splits.flatten()
ds_splits["train"][0]

{'prompt': '<s> bb: <s> bb: хавчаарт хавчаар <dat></s></s>',
 'answer': 'хавчаар <dat></s></s>'}

In [11]:
def preprocess_function(examples, tokenizer = tokenizer):
    return tokenizer(examples["prompt"])

tokenized_ds = ds_splits.map(
    preprocess_function,
    batched=True,
    num_proc=4,
    remove_columns=ds_splits["train"].column_names,
)

Map (num_proc=4):   0%|          | 0/14036 [00:00<?, ? examples/s]

Map (num_proc=4):   0%|          | 0/180 [00:00<?, ? examples/s]

Map (num_proc=4):   0%|          | 0/180 [00:00<?, ? examples/s]

In [13]:
block_size = 64

def group_texts(examples, block_size = block_size):
    # Concatenate all texts.
    concatenated_examples = {k: sum(examples[k], []) for k in examples.keys()}
    total_length = len(concatenated_examples[list(examples.keys())[0]])
    # We drop the small remainder, we could add padding if the model supported it instead of this drop, you can
    # customize this part to your needs.
    if total_length >= block_size:
        total_length = (total_length // block_size) * block_size
    # Split by chunks of block_size.
    result = {
        k: [t[i : i + block_size] for i in range(0, total_length, block_size)]
        for k, t in concatenated_examples.items()
    }
    result["labels"] = result["input_ids"].copy()
    return result

lm_dataset = tokenized_ds.map(group_texts, batched=True, num_proc=4)

Map (num_proc=4):   0%|          | 0/14036 [00:00<?, ? examples/s]

Map (num_proc=4):   0%|          | 0/180 [00:00<?, ? examples/s]

Map (num_proc=4):   0%|          | 0/180 [00:00<?, ? examples/s]

In [14]:
from transformers import DataCollatorForLanguageModeling

tokenizer.pad_token = tokenizer.eos_token
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

In [15]:
from transformers import AutoModelForCausalLM, TrainingArguments, Trainer

model = AutoModelForCausalLM.from_pretrained("bayartsogt/mongolian-gpt2")

config.json:   0%|          | 0.00/864 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/510M [00:00<?, ?B/s]

In [16]:
training_args = TrainingArguments(
    output_dir="my_awesome_gpt-inflection-model",
    evaluation_strategy="epoch",
    learning_rate=2e-5,
    weight_decay=0.01,
    push_to_hub=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=lm_dataset["train"],
    eval_dataset=lm_dataset["test"],
    data_collator=data_collator,
)

trainer.train()

d:\Anaconda3\envs\pt_env\Lib\site-packages\accelerate\accelerator.py:432: FutureWarning: Passing the following arguments to `Accelerator` is deprecated and will be removed in version 1.0 of Accelerate: dict_keys(['dispatch_batches', 'split_batches', 'even_batches', 'use_seedable_sampler']). Please pass an `accelerate.DataLoaderConfiguration` instead: 
dataloader_config = DataLoaderConfiguration(dispatch_batches=None, split_batches=False, even_batches=True, use_seedable_sampler=True)
  warnings.warn(
Failed to detect the name of this notebook, you can set it manually with the WANDB_NOTEBOOK_NAME environment variable to enable code saving.
wandb: Currently logged in as: temuujin-razy. Use `wandb login --relogin` to force relogin


  0%|          | 0/1461 [00:00<?, ?it/s]

  0%|          | 0/4 [00:00<?, ?it/s]

{'eval_loss': 3.4422738552093506, 'eval_runtime': 0.6264, 'eval_samples_per_second': 44.702, 'eval_steps_per_second': 6.386, 'epoch': 1.0}
{'loss': 2.1657, 'grad_norm': 1.078932762145996, 'learning_rate': 1.3155373032169748e-05, 'epoch': 1.03}


  0%|          | 0/4 [00:00<?, ?it/s]

{'eval_loss': 3.5148260593414307, 'eval_runtime': 0.3676, 'eval_samples_per_second': 76.16, 'eval_steps_per_second': 10.88, 'epoch': 2.0}
{'loss': 0.8793, 'grad_norm': 0.8629420399665833, 'learning_rate': 6.310746064339493e-06, 'epoch': 2.05}


  0%|          | 0/4 [00:00<?, ?it/s]

{'eval_loss': 3.5765459537506104, 'eval_runtime': 0.3182, 'eval_samples_per_second': 88.0, 'eval_steps_per_second': 12.571, 'epoch': 3.0}
{'train_runtime': 463.677, 'train_samples_per_second': 25.207, 'train_steps_per_second': 3.151, 'train_loss': 1.2990561399779688, 'epoch': 3.0}


TrainOutput(global_step=1461, training_loss=1.2990561399779688, metrics={'train_runtime': 463.677, 'train_samples_per_second': 25.207, 'train_steps_per_second': 3.151, 'train_loss': 1.2990561399779688, 'epoch': 3.0})

In [17]:
import math

eval_results = trainer.evaluate()
print(f"Perplexity: {math.exp(eval_results['eval_loss']):.2f}")

  0%|          | 0/4 [00:00<?, ?it/s]

Perplexity: 35.75


In [23]:
prompt = "<s> bb: сургуулийн"

from transformers import pipeline

generator = pipeline("text-generation", model=model, tokenizer = tokenizer, num_beams=5)

ans = generator(prompt, max_length = 20)

print(str(ans[0]['generated_text']))
print(str(ans))

<s> bb: сургуулийн сургууль <gen> bb:  bb
[{'generated_text': '<s> bb: сургуулийн сургууль <gen> bb:  bb'}]


In [24]:
prompt = "<s> bb: утааны"
ans = generator(prompt, max_length = 15)
print(str(ans))

[{'generated_text': '<s> bb: утааны утаа <gen> bb'}]
